### ライブラリの読み込み

In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb

from sklearn.model_selection import KFold, GridSearchCV
from sklearn.utils import resample
from collections import defaultdict

### データの読み込み

In [2]:
# TCVファイルの読み込み
train_data = pd.read_csv('../000.data/train/train_A.tsv', sep="\t")
test = pd.read_csv('../000.data/test.tsv', sep="\t")

In [3]:
# 末尾が "_A" のテストデータだけを抽出
test_data = test[test["user_id"].str.endswith("_A")]

In [4]:
# 関連度スコアの設定
def compute_relevance(row):
    if row["event_type"] == 3 and row["ad"] == 1:  # 広告経由の購入
        return 3
    elif row["event_type"] == 2:  # 広告クリック
        return 2
    elif row["event_type"] == 1:  # 詳細ページ閲覧
        return 1
    else:  # カート追加
        return 0

train_data["relevance"] = train_data.apply(compute_relevance, axis=1)

### 特徴量エンジニアリング

In [5]:
# タイムスタンプをdatetime型に変換
train_data["time_stamp"] = pd.to_datetime(train_data["time_stamp"])

In [6]:
# 最新の行動からの経過時間（秒）
train_data["time_since_last"] = (train_data["time_stamp"].max() - train_data["time_stamp"]).dt.total_seconds()

In [7]:
# ユーザーごとの行動回数集計
user_features = train_data.groupby("user_id").agg(
    user_total_interactions=("product_id", "count"),
    user_unique_products=("product_id", "nunique"),
    user_avg_time_since=("time_since_last", "mean"),
    user_click_rate=("relevance", lambda x: np.mean(x >= 2))
).reset_index()

In [8]:
# 商品ごとの人気度
product_features = train_data.groupby("product_id").agg(
    product_total_interactions=("user_id", "count"),
    product_unique_users=("user_id", "nunique"),
    product_purchase_rate=("relevance", lambda x: np.mean(x == 3))
).reset_index()

In [9]:
# データ結合
train_data = train_data.merge(user_features, on="user_id", how="left")
train_data = train_data.merge(product_features, on="product_id", how="left")

### XGBoost 用データセットの準備 ～ モデル学習 ～ nGCDの計算

In [10]:
# 学習用特徴量
features = [
    "user_total_interactions", "user_unique_products", "user_avg_time_since",
    "user_click_rate", "product_total_interactions", "product_unique_users",
    "product_purchase_rate"
]

In [11]:
# クラス不均衡に対する学習データのバランス調整
train_data_pos = train_data[train_data["relevance"] >= 2]  # 広告クリックと購入のデータ
train_data_neg = train_data[train_data["relevance"] < 2]   # その他のデータ

In [12]:
# 少数派クラス（購入・広告クリック）をオーバーサンプリング
train_data_pos_oversampled = resample(train_data_pos, 
                                      replace=True, 
                                      n_samples=len(train_data_neg),  # 多数派クラスのサンプル数に合わせる
                                      random_state=42)

In [13]:
# サンプリング後のデータ
train_data_balanced = pd.concat([train_data_neg, train_data_pos_oversampled])

In [14]:
# ハイパーパラメータの自動調整とモデル作成
best_params = None
best_score = -np.inf

for max_depth in [6, 8, 10]:
    for min_child_weight in [1, 5, 10]:
        for subsample in [0.8, 0.9]:
            for colsample_bytree in [0.7, 0.8]:
                for eta in [0.05, 0.1]:
                    for gamma in [0, 0.2]:
                        model = xgb.XGBRanker(
                            objective="rank:ndcg",
                            eval_metric="ndcg",
                            booster="gbtree",
                            max_depth=max_depth,
                            min_child_weight=min_child_weight,
                            subsample=subsample,
                            colsample_bytree=colsample_bytree,
                            eta=eta,
                            gamma=gamma,
                        )

                        X_train = train_data_balanced[features]
                        y_train = train_data_balanced["relevance"]
                        group_train = train_data_balanced.groupby("user_id").size().to_numpy()

                        dtrain = xgb.DMatrix(X_train, label=y_train)
                        model.fit(X_train, y_train, group=group_train, verbose=False)

                        ndcg_score = float(model.get_booster().eval(dtrain, iteration=0).split(":")[1])

                        if ndcg_score > best_score:
                            best_score = ndcg_score
                            best_params = {
                                "max_depth": max_depth,
                                "min_child_weight": min_child_weight,
                                "subsample": subsample,
                                "colsample_bytree": colsample_bytree,
                                "eta": eta,
                                "gamma": gamma
                            }

print("最適パラメータ:", best_params)
print("最適スコア:", best_score)

最適パラメータ: {'max_depth': 8, 'min_child_weight': 1, 'subsample': 0.9, 'colsample_bytree': 0.7, 'eta': 0.1, 'gamma': 0.2}
最適スコア: 0.9999999999999998


### 予測と提出データの作成

In [15]:
# 推薦候補商品リストを作成
candidate_products = train_data.groupby("product_id")["user_id"].count().reset_index()
candidate_products = candidate_products.sort_values("user_id", ascending=False).head(22)  # 上位22商品

In [16]:
# すべてのユーザーに候補商品を紐づける
test_data = test_data.assign(key=1).merge(candidate_products.assign(key=1), on="key").drop("key", axis=1)
test_data = test_data.rename(columns={'user_id_x': 'user_id'}).drop(columns=['user_id_y'])

In [17]:
# 評価データの前処理（学習データと同じ特徴量処理を適用）
test_data = test_data.merge(user_features, on="user_id", how="left")
test_data = test_data.merge(product_features, on="product_id", how="left")

In [18]:
# 欠損値処理（初めてのユーザーや商品に対して）
test_data.fillna(0, inplace=True)

In [19]:
# 予測用データ
X_test = test_data[features]

In [20]:
# 最適モデルを使用して予測
model = xgb.XGBRanker(**best_params)
model.fit(X_train, y_train, group=group_train)
test_data["score"] = model.predict(X_test)

In [21]:
# 各ユーザーごとにランキング付け（スコアが高い順）
test_data["rank"] = test_data.groupby("user_id")["score"].rank(ascending=False, method="first")

In [22]:
# nDCG の計算
def dcg_at_k(relevance_scores, k):
    relevance_scores = np.asarray(relevance_scores)[:k]
    return np.sum(relevance_scores / np.log2(np.arange(2, relevance_scores.size + 2)))

In [23]:
def ndcg_at_k(relevance_scores, k):
    dcg = dcg_at_k(relevance_scores, k)
    ideal_dcg = dcg_at_k(sorted(relevance_scores, reverse=True), k)
    return dcg / ideal_dcg if ideal_dcg > 0 else 0

In [24]:
def evaluate_ndcg(data, ground_truth, k=22):
    ndcg_scores = []
    for user_id, group in data.groupby("user_id"):
        predicted_items = group.sort_values("score", ascending=False)["product_id"].tolist()
        relevance_scores = [ground_truth[user_id].get(item, 0) for item in predicted_items]
        ndcg_scores.append(ndcg_at_k(relevance_scores, k))
    return np.mean(ndcg_scores) if ndcg_scores else 0

In [25]:
# テストデータの nDCG 計算
ground_truth_test = defaultdict(dict)
for _, row in train_data.iterrows():
    ground_truth_test[row["user_id"]][row["product_id"]] = row["relevance"]

ndcg_test = evaluate_ndcg(test_data, ground_truth_test, k=22)
print(f"Test nDCG@22: {ndcg_test:.4f}")

Test nDCG@22: 0.2581


In [26]:
# 提出用データの作成（上位 22 件のみ）
submission = test_data[test_data["rank"] <= 22][["user_id", "product_id", "rank"]]

In [27]:
# rankを整数に変換
submission['rank'] = submission['rank'].astype(int)

In [28]:
# 保存
submission.to_csv('./predict_file/predictions_A.tsv', sep="\t", index=False)